## TZ ref admixture - plotting

In [34]:
!pip install -q malariagen_data

In [ ]:
!pip install malariagen_data "xyzservices < 2023.10.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 101.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.5/302.5 kB 33.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.7/138.7 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 66.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.9/206.9 kB 22.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 43.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.0 MB/s eta

In [35]:
import malariagen_data
import dask.array as da
import allel
import numpy as np
import plotly.express as px
import pandas as pd
from dask.diagnostics import ProgressBar
import google.colab
import os

In [ ]:
google.colab.drive.mount("drive")

Mounted at drive


In [ ]:
results_dir = "drive/MyDrive/Tanzania/TZ-ref-admixture-131123"

In [36]:
ag3 = malariagen_data.Ag3()

In [ ]:
def get_plink_file_path(
    contig,
    n_snps,
    min_minor_ac,
    thin_offset,
    max_missing_an,
):
    return f"{results_dir}/{contig}.{n_snps}.{min_minor_ac}.{thin_offset}.{max_missing_an}"


In [ ]:
#set seed

In [ ]:
def load_admixture(
    contig,
    n_snps,
    thin_offset,
    min_minor_ac,
    K,
    seed=42,
):
    contig_label = contig.replace(",", ".")
    contig_label = f"chr{contig_label}"
    plink_file_path = get_plink_file_path(
        contig=contig_label, n_snps=n_snps, min_minor_ac=min_minor_ac, thin_offset=thin_offset, max_missing_an=0
    )

    fam_file_path = f"{plink_file_path}.fam"
    admixture_dir = f"{plink_file_path}.admixture/{seed}"
    Q_file_path = f"{admixture_dir}/{K}.Q"
    P_file_path = f"{admixture_dir}/{K}.P"
    log_file_path = f"{admixture_dir}/{K}.log"

    # load results - sample IDs
    df_fam = pd.read_csv(
        fam_file_path,
        sep=" ",
        header=None,
        names=["family_id", "sample_id", "father", "mother", "sex", "phenotype"],
        index_col=False,
    )
    samples = df_fam["sample_id"]
    df_samples = ag3.sample_metadata()
    df_samples = (
        df_samples
        .set_index("sample_id")
        .loc[samples]
    )

    # load results - ancestry fractions
    df_q = pd.read_csv(
        Q_file_path,
        sep=" ",
        header=None,
        names=[f"pop{i}" for i in range(K)],
        index_col=False,
    )
    df_q["popmax"] = df_q.idxmax(axis="columns")
    df_q["popmax_frac"] = df_q.apply(lambda row: row[row["popmax"]], axis="columns")
    df_q.set_index(samples, inplace=True)

    df_out = df_q.join(df_samples).reset_index()
    df_out.attrs["K"] = K
    return df_out


In [ ]:
df = load_admixture(
    contig="3L:15,000,000-41,000,000",
    n_snps=50_000,
    thin_offset=0,
    min_minor_ac=5,
    K=2,
)
df.head()

,sample_id,pop0,pop1,popmax,popmax_frac,partner_sample_id,contributor,country,location,year,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,AB0085-Cx,0.999990,0.000010,pop0,0.999990,BF2-4,Austin Burt,Burkina Faso,Pala,2012,...,Hauts-Bassins,BF-09,Houet,gambiae,BF-09_gamb_2012,BF-09_gamb_2012_07,BF-09_gamb_2012_Q3,BF-09_Houet_gamb_2012,BF-09_Houet_gamb_2012_07,BF-09_Houet_gamb_2012_Q3
1,AB0086-Cx,0.994438,0.005562,pop0,0.994438,BF2-6,Austin Burt,Burkina Faso,Pala,2012,...,Hauts-Bassins,BF-09,Houet,gambiae,BF-09_gamb_2012,BF-09_gamb_2012_07,BF-09_gamb_2012_Q3,BF-09_Houet_gamb_2012,BF-09_Houet_gamb_2012_07,BF-09_Houet_gamb_2012_Q3
2,AB0087-C,0.999990,0.000010,pop0,0.999990,BF3-3,Austin Burt,Burkina Faso,Bana Village,2012,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
3,AB0088-C,0.999990,0.000010,pop0,0.999990,BF3-5,Austin Burt,Burkina Faso,Bana Village,2012,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
4,AB0089-Cx,0.999990,0.000010,pop0,0.999990,BF3-8,Austin Burt,Burkina Faso,Bana Village,2012,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3


In [ ]:
df['cohort_admin1_year'].unique()

array(['BF-09_gamb_2012', 'BF-09_colu_2012', 'BF-09_colu_2014',
       'BF-09_gamb_2014', 'BF-07_gamb_2004', 'KE-14_arab_2012',
       'KE-14_gcx3_2012', 'KE-14_gamb_2000', 'KE-14_arab_2007',
       'TZ-05_gcx3_2015', 'TZ-05_arab_2015', 'TZ-05_gamb_2015',
       'TZ-26_arab_2012', 'TZ-13_arab_2012', 'TZ-25_gamb_2013',
       'TZ-25_gcx3_2013', 'TZ-25_arab_2013', 'UG-E_arab_2012',
       'UG-E_gamb_2012', 'UG-W_gamb_2012', 'UG-W_arab_2012'], dtype=object)

In [ ]:
import plotly.graph_objects as go
import plotly.subplots
import plotly.express as px

In [ ]:
# plotly.subplots.make_subplots?

In [ ]:
def plot_admixture(
    contig,
    n_snps,
    thin_offset,
    min_minor_ac,
    by = 'cohort_admin1_year',
    col_keys = None,
    k_vals=list(range(2, 7)),
    colors=px.colors.qualitative.Plotly,
    width=900,
    height=900,
):

    # setup
    #read fam file for sample ids
    contig_label = contig.replace(",", ".")
    contig_label = f"chr{contig_label}"
    plink_file_path = get_plink_file_path(contig=contig_label, n_snps=n_snps, thin_offset=0, min_minor_ac=min_minor_ac, max_missing_an=0)
    fam_file_path = f"{plink_file_path}.fam"
    df_fam = pd.read_csv(
            fam_file_path,
            sep=" ",
            header=None,
            names=["family_id", "sample_id", "father", "mother", "sex", "phenotype"],
            index_col=False,
        )
    samples = df_fam["sample_id"]
    samples = samples.tolist()


    df_samples = ag3.sample_metadata(sample_query=f'sample_id in {samples}')

    #allow change of order. This could probably be coded better.
    if col_keys is not None:
      col_keys = col_keys
    else:
      col_keys = df_samples[by].unique().tolist()

    rows = len(k_vals)
    cols = len(col_keys)
    col_widths = df_samples[by].value_counts().loc[col_keys].to_list()

    # create figure
    fig = plotly.subplots.make_subplots(
        rows=rows,
        cols=cols,
        shared_yaxes=True,
        column_titles=col_keys,
        row_titles=[f"K={K}" for K in k_vals],
        column_widths=col_widths,
        x_title="Samples",
        y_title=f"Ancestry fraction",
        horizontal_spacing=.01,
        vertical_spacing=.02,
    )

    for i, K in enumerate(k_vals):

        df = load_admixture(
            contig=contig,
            n_snps=n_snps,
            thin_offset=thin_offset,
            min_minor_ac=min_minor_ac,
            K=K,
        )
        data_frame = df.sort_values(by=["popmax", "popmax_frac"], ascending=False)
        K = data_frame.attrs["K"]
        admx_cols = [f"pop{i}" for i in range(K)]
        g = data_frame.groupby(by=by)

        for j, key in enumerate(col_keys):
            jx = g.groups[key]
            dfk = data_frame.loc[jx]
            x = "sample_id"
            for y, color in zip(admx_cols, colors):
                fig.add_trace(
                    go.Bar(
                        x=dfk["sample_id"],
                        y=dfk[y],
                        marker_color=color,
                        name=y,
                    ),
                    row=i+1, col=j+1,
                )

    fig.update_yaxes(
        range=(0, 1)
    )

    fig.update_xaxes(
        tickmode='array',
        tickvals=[],
    )

    fig.update_yaxes(
        tickmode='array',
        tickvals=[],
    )

    fig.update_layout(
        barmode="stack",
        bargap=0,
        width=width,
        height=height,
        showlegend=False,
        title_text=None,
    )

    fig.update_traces(
        marker=dict(line=dict(width=0)),
    )

    # rotate all the subtitles using len of cohorts
    for annotation in fig['layout']['annotations'][0:cols]:
        annotation['textangle']=-25
        annotation['xanchor'] = 'left'




    return fig

In [37]:
df_samples = ag3.sample_metadata(sample_query = 'taxon in ["gambiae","coluzzii"] and country == "Burkina Faso" or taxon in ["gambiae","coluzzii","arabiensis","gcx3"] and country == "Uganda" or country == "Tanzania" or country == "Kenya"')

In [38]:
df_samples.groupby(['country','taxon','cohort_admin1_year','location']).size()

country       taxon       cohort_admin1_year  location      
Burkina Faso  coluzzii    BF-09_colu_2012     Bana Village       42
                                              Pala               11
                                              Souroukoudinga     29
                          BF-09_colu_2014     Bana Village       47
                                              Souroukoudinga      6
              gambiae     BF-07_gamb_2004     Monomtenga         13
                          BF-09_gamb_2012     Bana Village       23
                                              Pala               48
                                              Souroukoudinga     28
                          BF-09_gamb_2014     Bana Village       15
                                              Pala               16
                                              Souroukoudinga     15
Kenya         arabiensis  KE-14_arab_2007     Kilifi              3
                          KE-14_arab_2012     Kilifi   

In [ ]:
df_tz_samples = ag3.sample_metadata(sample_query = 'country == "Tanzania"')

In [ ]:
df_tz_samples.groupby(['country','taxon','cohort_admin1_year','location']).size()

country   taxon       cohort_admin1_year  location
Tanzania  arabiensis  TZ-05_arab_2015     Muleba      137
                      TZ-13_arab_2012     Tarime       47
                      TZ-25_arab_2013     Muheza        1
                      TZ-26_arab_2012     Moshi        40
          gambiae     TZ-05_gamb_2015     Muleba       32
                      TZ-25_gamb_2013     Muheza       32
          gcx3        TZ-05_gcx3_2015     Muleba        1
                      TZ-25_gcx3_2013     Muheza       10
dtype: int64

In [ ]:
#Muheza similar to Kenya
fig = plot_admixture(
    contig="3L:15,000,000-41,000,000",
    n_snps=50_000,
    thin_offset=0,
    min_minor_ac=5,
    by='cohort_admin1_year',
    col_keys=['UG-E_arab_2012','KE-14_arab_2007','UG-W_arab_2012','KE-14_arab_2012','TZ-13_arab_2012','TZ-26_arab_2012','TZ-25_arab_2013','TZ-05_arab_2015',
              'BF-09_colu_2012', 'BF-09_colu_2014',
              'BF-09_gamb_2012','BF-09_gamb_2014', 'BF-07_gamb_2004','UG-E_gamb_2012','UG-W_gamb_2012','KE-14_gamb_2000','TZ-25_gamb_2013','TZ-05_gamb_2015',
              'KE-14_gcx3_2012', 'TZ-25_gcx3_2013','TZ-05_gcx3_2015',
              ],
    )
fig

In [ ]:
fig = plot_admixture(
    contig="3L:15,000,000-41,000,000",
    n_snps=50_000,
    thin_offset=0,
    min_minor_ac=5,
    by='taxon',
    col_keys=['arabiensis',
              'coluzzii',
              'gambiae',
              'gcx3'],
)
fig

In [ ]:
contig="3L:15,000,000-41,000,000"
contig_label = contig.replace(",", ".")
contig_label = f"chr{contig_label}"
get_plink_file_path(
    contig=contig_label,
    n_snps=50_000,
    thin_offset=0,
    min_minor_ac=5,
    max_missing_an=0,
)

'drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0'

In [ ]:
!grep CV drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42/*.log

drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42/2.log:CV error (K=2): 0.11539
drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42/3.log:CV error (K=3): 0.10991
drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42/4.log:CV error (K=4): 0.10800
drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42/5.log:CV error (K=5): 0.10634
drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42/6.log:CV error (K=6): 0.10684


In [ ]:
def extract_cv_error(
    contig,
    n_snps,
    thin_offset,
    min_minor_ac,
    k_vals=list(range(2, 7)),
    seed=42,
):
    contig_label = contig.replace(",", ".")
    contig_label = f"chr{contig_label}"
    plink_file_path = get_plink_file_path(
        contig=contig_label,
        n_snps=n_snps,
        thin_offset=thin_offset,
        min_minor_ac=min_minor_ac,
        max_missing_an=0,
    )
    admixture_path = f"{plink_file_path}.admixture/{seed}"
    cv_error = []
    for k in k_vals:
        log_path = f"{admixture_path}/{k}.log"
        with open(log_path, mode="rt") as log_file:
            for line in log_file.readlines():
                if line.startswith("CV error"):
                    e = float(line.split(":")[1])
                    cv_error.append(e)
    return pd.DataFrame({"contig": contig, "seed": seed, "k": k_vals, "cv_error": cv_error})



In [ ]:
df_cv_error = extract_cv_error(
                              contig="3L:15,000,000-41,000,000",
                              n_snps=50_000,
                              thin_offset=0,
                              min_minor_ac=5,
                              )

In [ ]:
fig = px.line(
    data_frame=df_cv_error,
    x="k",
    y="cv_error",
    line_group="contig",
    color="contig",
    markers=True,
    labels={
        'k': 'K',
        'cv_error': "Cross-validation error",
        "contig": "Contig",
    },
    width=600,
    height=400,
)

fig.update_xaxes(
    tickmode='array',
    tickvals=[2, 3, 4, 5, 6],
)


In [ ]:
fig = px.scatter(
    data_frame=df_cv_error_reps,
    x="k",
    y="cv_error",
    color="contig",
    labels={
        'k': 'K',
        'cv_error': "Cross-validation error",
        "contig": "Contig",
    },
    width=600,
    height=400,
)

fig.update_xaxes(
    tickmode='array',
    tickvals=[2, 3, 4, 5, 6],
)


In [ ]:
fig = px.violin(
    data_frame=df_cv_error_reps,
    x="k",
    y="cv_error",
    # color="contig",
    points="all",
    labels={
        'k': 'K',
        'cv_error': "Cross-validation error",
        "contig": "Contig",
    },
    width=600,
    height=400,
)

fig.update_xaxes(
    tickmode='array',
    tickvals=[2, 3, 4, 5, 6],
)

fig.update_traces(
    pointpos=0,
)

In [ ]:
#combine multiple runs with pong.

In [ ]:
!pip install pong

In [ ]:
!pong -h

usage: pong [-h] -m FILEMAP [-c IGNORE_COLS] [-o OUTPUT_DIR] [-i IND2POP] [-n POP_NAMES]
            [-l COLOR_LIST] [-f] [-s SIM_THRESHOLD] [--col_delim COL_DELIM]
            [--dist_metric DIST_METRIC] [--disable_server] [-p PORT] [-v] [-g]

-------------------------------- pong, v1.5 --------------------------------

options:
  -h, --help            show this help message and exit
  -m FILEMAP, --filemap FILEMAP
                        path to params file containing information about input Q-matrix files
  -c IGNORE_COLS, --ignore_cols IGNORE_COLS
                        ignore the first i columns of every data line. Typically 5 for Structure
                        output and 0 for ADMIXTURE output. Default = 0
  -o OUTPUT_DIR, --output_dir OUTPUT_DIR
                        specify output dir for files to be written to. By default, pong makes a
                        folder named "pong_output_datetime" in the current working directory,
                        where "datetime" is

In [ ]:
df_samples=df_samples.reset_index()

In [ ]:
#setup pong

In [ ]:
contig="3L:15,000,000-41,000,000"
contig_label = contig.replace(",", ".")
contig_label = f"chr{contig_label}"
plink_file_path = get_plink_file_path(
    contig=contig_label,
    n_snps=50_000,
    thin_offset=0,
    min_minor_ac=5,
    max_missing_an=0,
)


In [ ]:
plink_file_path

'drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.5000.5.0.0'

In [ ]:
#get mapfile
cola = []
colb = []
for seed in range(0,2):
  for k in range(2,8):
    cola.append(f'{seed}_{k}')
    colb.append(f'{plink_file_path}.admixture/{seed}/{k}.Q')

In [ ]:
#get mapfile
cola = []
colb = []
for k in range(2,8):
  cola.append(f'{seed}_{k}')
  colb.append(f'{plink_file_path}.admixture/1/{k}.Q')

In [ ]:
mapfile = pd.DataFrame({'a': cola, 'b': colb})

In [ ]:
mapfile_txt = mapfile.to_csv(f'{plink_file_path}.admixture/mapfile.txt',sep='\t',index=False, header=False)

In [ ]:
ind2pop=df_samples['cohort_admin1_year']

In [ ]:
ind2pop_txt = ind2pop.to_csv(f'{plink_file_path}.admixture/ind2pop.txt',sep='\t',index=False, header=False)

In [ ]:
#get pop order
pop_order=ind2pop.unique()

In [ ]:
poporder = pd.DataFrame({'a': pop_order})

In [ ]:
poporder

,a
0,TZ-13_arab_2012
1,TZ-05_arab_2015
2,TZ-26_arab_2012
3,KE-14_arab_2012
4,KE-14_gamb_2000
5,TZ-25_gamb_2013
6,TZ-05_gamb_2015
7,KE-14_gcx3_2012
8,TZ-25_gcx3_2013


In [ ]:
poporder_txt = poporder.to_csv(f'{plink_file_path}.admixture/poporder.txt',sep='\t',index=False, header=False)

In [ ]:
!pong -m {plink_file_path}.admixture/mapfile.txt -i {plink_file_path}.admixture/ind2pop.txt -n {plink_file_path}.admixture/poporder.txt -o {plink_file_path}/pong


Output dir pong already exists.
Overwrite? (y/n): y

-------------------------------------------------------------------
                            p o n g
      by A. Behr, K. Liu, T. Devlin, G. Liu-Fang, and S. Ramachandran
                       Version 1.5 (2021)
-------------------------------------------------------------------
-------------------------------------------------------------------

Parsing input and generating cluster network graph
Traceback (most recent call last):
  File "/usr/local/bin/pong", line 8, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.10/dist-packages/pong/main.py", line 295, in main
    run_pong(*run_pong_args)
  File "/usr/local/lib/python3.10/dist-packages/pong/main.py", line 323, in run_pong
    parse.parse_multicluster_input(pongdata, pong_filemap, opts.ignore_cols, 
  File "/usr/local/lib/python3.10/dist-packages/pong/parse.py", line 30, in parse_multicluster_input
    qfiles_info = np.genfromtxt(filemap, delimiter='\t', 
  Fi

In [ ]:
df_cv_error

,contig,seed,k,cv_error
0,KB663610,42,2,0.19747
1,KB663610,42,3,0.19911
2,KB663610,42,4,0.20564
3,KB663610,42,5,0.21316
4,KB663610,42,6,0.22093
5,KB663721,42,2,0.19633
6,KB663721,42,3,0.20255
7,KB663721,42,4,0.20453
8,KB663721,42,5,0.21180
9,KB663721,42,6,0.21928


In [ ]:
df_cv_error.iloc[df_cv_error.groupby("contig")["cv_error"].idxmin()]

,contig,seed,k,cv_error
0,KB663610,42,2,0.19747
45,KB663633,42,2,0.20730
5,KB663721,42,2,0.19633
10,KB663832,42,2,0.20910
15,KB663943,42,2,0.20335
20,KB664054,42,2,0.19547
25,KB664165,42,2,0.19048
30,KB664255,42,2,0.18733
35,KB664266,42,2,0.19597
41,KB664277,42,3,0.22946


In [ ]:
df_cv_error_reps.iloc[df_cv_error_reps.groupby(["contig", "seed"])["cv_error"].idxmin()]

,contig,seed,k,cv_error
0,KB663610,42,2,0.19747
5,KB663610,64000,2,0.19731
90,KB663633,42,2,0.20730
95,KB663633,64000,2,0.20720
10,KB663721,42,2,0.19633
15,KB663721,64000,2,0.19630
20,KB663832,42,2,0.20910
25,KB663832,64000,2,0.20915
30,KB663943,42,2,0.20335
35,KB663943,64000,2,0.20339
